# ReefScan-ID: Stage 2 patch dataset

Patches for stage 2 of the cascade: points inside coral, labelled healthy / bleached / dead. Run in Google Colab; this notebook needs no GPU.

### Setup: Colab runtime and Google Drive

In [1]:
import os
from google.colab import drive

drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/ReefScan-ID"
raw_dir = base_dir + "/RawData"
print("Drive folder found: {}".format(os.path.isdir(base_dir)))
print("Cache folder exists: {}".format(os.path.isdir(raw_dir)))
print("CPU count: {}".format(os.cpu_count()))

Mounted at /content/drive
Drive folder found: True
Cache folder exists: True
CPU count: 2


### Setup: libraries

In [2]:
!pip install -q datasets

import datasets
import numpy as np
import pandas as pd

print("datasets version: {}".format(datasets.__version__))
print("numpy version: {}".format(np.__version__))
print("pandas version: {}".format(pd.__version__))

datasets version: 4.8.5
numpy version: 2.4.6
pandas version: 2.2.3


## Section 1: Stage 2 label map

Question: which Coralscapes classes become healthy, bleached and dead for stage 2, and how many pixels does each of the three hold in the train and validation splits?

Checks: map every mask value to a stage 2 state, print the classes in each state, count the pixels of every mask value in train and validation, then print the share of each state. The test split is left out here because it is kept for the final evaluation.

### Load the dataset

In [3]:
import time
from datasets import load_dataset

start = time.time()
ds = load_dataset("EPFL-ECEO/coralscapes", cache_dir=raw_dir)
print(ds)
print("Loading took {:.1f} seconds".format(time.time() - start))

README.md:   0%|          | 0.00/13.7k [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 1517
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 166
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 392
    })
})
Loading took 53.7 seconds


### Class names from the dataset card

Ids and names copied from the table on the Coralscapes dataset card (read 2026-11-12), as in notebook 02.

In [4]:
class_names = {
    1: "seagrass",
    2: "trash",
    3: "other coral dead",
    4: "other coral bleached",
    5: "sand",
    6: "other coral alive",
    7: "human",
    8: "transect tools",
    9: "fish",
    10: "algae covered substrate",
    11: "other animal",
    12: "unknown hard substrate",
    13: "background",
    14: "dark",
    15: "transect line",
    16: "massive/meandering bleached",
    17: "massive/meandering alive",
    18: "rubble",
    19: "branching bleached",
    20: "branching dead",
    21: "millepora",
    22: "branching alive",
    23: "massive/meandering dead",
    24: "clam",
    25: "acropora alive",
    26: "sea cucumber",
    27: "turbinaria",
    28: "table acropora alive",
    29: "sponge",
    30: "anemone",
    31: "pocillopora alive",
    32: "table acropora dead",
    33: "meandering bleached",
    34: "stylophora alive",
    35: "sea urchin",
    36: "meandering alive",
    37: "meandering dead",
    38: "crown of thorn",
    39: "dead clam",
}

print("Classes on the card: {}".format(len(class_names)))

Classes on the card: 39


### Stage 2 label map

In [5]:
healthy_ids = [6, 17, 22, 25, 28, 31, 34, 36]
bleached_ids = [4, 16, 19, 33]
dead_ids = [3, 20, 23, 32, 37]
excluded_ids = [21, 27]

state_names = {0: "healthy", 1: "bleached", 2: "dead"}
excluded_state = 253
not_coral_state = 255

# Values that are not coral stay at the not coral code
state_map = np.full(256, not_coral_state, dtype=np.uint8)
state_map[healthy_ids] = 0
state_map[bleached_ids] = 1
state_map[dead_ids] = 2
state_map[excluded_ids] = excluded_state

for state_id, state_name in state_names.items():
    names = [class_names[v] for v in class_names if state_map[v] == state_id]
    print("{} ({} ids): {}".format(state_name, len(names), names))
excluded = [class_names[v] for v in class_names if state_map[v] == excluded_state]
print("excluded coral ({} ids): {}".format(len(excluded), excluded))
print("Ids that are not coral: {}".format(sum(1 for v in class_names if state_map[v] == not_coral_state)))

healthy (8 ids): ['other coral alive', 'massive/meandering alive', 'branching alive', 'acropora alive', 'table acropora alive', 'pocillopora alive', 'stylophora alive', 'meandering alive']
bleached (4 ids): ['other coral bleached', 'massive/meandering bleached', 'branching bleached', 'meandering bleached']
dead (5 ids): ['other coral dead', 'branching dead', 'massive/meandering dead', 'table acropora dead', 'meandering dead']
excluded coral (2 ids): ['millepora', 'turbinaria']
Ids that are not coral: 20


### Count pixels per mask value

In [6]:
pixel_counts = {}
for split in ["train", "validation"]:
    counts = np.zeros(256, dtype=np.int64)
    masks = ds[split].select_columns(["label"])
    for row in masks:
        mask = np.array(row["label"])
        counts += np.bincount(mask.ravel(), minlength=256)
    pixel_counts[split] = counts
    print("{}: {} masks counted, {} pixels".format(split, masks.num_rows, counts.sum()))

train: 1517 masks counted, 3181379584 pixels
validation: 166 masks counted, 348127232 pixels


### Pixel share per stage 2 state

In [7]:
for split in ["train", "validation"]:
    counts = pixel_counts[split]
    total = counts.sum()
    state_pixels = {state_id: counts[state_map == state_id].sum() for state_id in state_names}
    labelled = sum(state_pixels.values())
    for state_id, state_name in state_names.items():
        print("{} {}: {:.2f}% of all pixels, {:.2f}% of the stage 2 coral pixels".format(
            split, state_name, 100 * state_pixels[state_id] / total, 100 * state_pixels[state_id] / labelled))
    excluded_pixels = counts[state_map == excluded_state].sum()
    print("{} excluded coral: {:.2f}% of all pixels".format(split, 100 * excluded_pixels / total))

train healthy: 21.19% of all pixels, 84.17% of the stage 2 coral pixels
train bleached: 0.83% of all pixels, 3.28% of the stage 2 coral pixels
train dead: 3.16% of all pixels, 12.55% of the stage 2 coral pixels
train excluded coral: 0.43% of all pixels
validation healthy: 10.67% of all pixels, 55.57% of the stage 2 coral pixels
validation bleached: 1.09% of all pixels, 5.69% of the stage 2 coral pixels
validation dead: 7.43% of all pixels, 38.73% of the stage 2 coral pixels
validation excluded coral: 0.38% of all pixels


**Observation**: The 39 classes were successfully mapped into healthy (8), bleached (4), and dead (5), with Millepora and Turbinaria excluded. Bleached coral represents only 3.28% of labelled train coral pixels, making it the minority class. Validation also has a different class distribution, with a higher proportion of dead coral (38.73% vs. 12.55% in training).

## Section 2: Sampling measurements

Question: how many candidate points does a stratified grid give per state and split, and how pure are the patches around them at each crop size?

Checks: put one random point in every grid cell of every train and validation photo, keep the points whose centre pixel is healthy, bleached or dead, and measure the purity and coral coverage of the window around each point at three crop sizes. Then count the points per state, show what a cap per state per photo keeps, and show what each purity threshold keeps. No patch is cut or saved here, and the test split is not used.

### Grid settings for the measurement

The values here are trial values for the tables below, not the final settings.

In [ ]:
grid_cell = 128
crop_sizes = [64, 128, 256]
seed = 0

photo_height = 1024
photo_width = 2048
margin = max(crop_sizes) // 2
cell_rows = (photo_height - 2 * margin) // grid_cell
cell_cols = (photo_width - 2 * margin) // grid_cell

print("Grid cell: {} px, crop sizes: {}".format(grid_cell, crop_sizes))
print("Margin from the photo edge: {} px".format(margin))
print("Grid cells per photo: {} rows x {} columns = {}".format(cell_rows, cell_cols, cell_rows * cell_cols))

### Measure points on train and validation

In [ ]:
def integral_table(flags):
    table = np.zeros((flags.shape[0] + 1, flags.shape[1] + 1), dtype=np.int64)
    table[1:, 1:] = flags.cumsum(axis=0).cumsum(axis=1)
    return table


def window_sum(table, tops, lefts, size):
    return (table[tops + size, lefts + size] - table[tops, lefts + size]
            - table[tops + size, lefts] + table[tops, lefts])


cell_count = cell_rows * cell_cols
records = []
for split in ["train", "validation"]:
    start = time.time()
    masks = ds[split].select_columns(["label"])
    for photo, row in enumerate(masks):
        state = state_map[np.array(row["label"])]

        # One random point in every grid cell
        rng = np.random.default_rng(seed + photo)
        ys = margin + np.repeat(np.arange(cell_rows), cell_cols) * grid_cell + rng.integers(0, grid_cell, cell_count)
        xs = margin + np.tile(np.arange(cell_cols), cell_rows) * grid_cell + rng.integers(0, grid_cell, cell_count)

        centre = state[ys, xs].astype(np.int64)
        keep = centre <= 2
        ys, xs, centre = ys[keep], xs[keep], centre[keep]
        if len(ys) == 0:
            continue

        tables = [integral_table(state == state_id) for state_id in state_names]
        record = {"split": split, "photo": photo, "y": ys, "x": xs, "state": centre}
        for size in crop_sizes:
            tops = ys - size // 2
            lefts = xs - size // 2
            sums = np.stack([window_sum(table, tops, lefts, size) for table in tables])
            coral = sums.sum(axis=0)
            same = sums[centre, np.arange(len(centre))]
            record["purity_{}".format(size)] = same / coral
            record["coverage_{}".format(size)] = coral / (size * size)
        records.append(pd.DataFrame(record))
    print("{}: {} photos measured in {:.1f} seconds".format(split, masks.num_rows, time.time() - start))

points = pd.concat(records, ignore_index=True)
points["state_name"] = points["state"].map(state_names)
print("Points kept: {}".format(len(points)))
print(points.head(5).to_string())

### Points per state and split

In [ ]:
print("Points per state and split:")
print(pd.crosstab(points["split"], points["state_name"]))

per_photo = points.groupby(["split", "photo", "state_name"]).size().rename("points").reset_index()
summary = per_photo.groupby(["split", "state_name"])["points"].describe(percentiles=[0.5, 0.9])
summary = summary[["count", "50%", "90%", "max"]]
summary.columns = ["photos with points", "median per photo", "90th percentile", "max per photo"]
print("Points per photo, for photos that have the state:")
print(summary.to_string())

### What a cap per state per photo keeps

In [ ]:
caps = [5, 10, 20, 40]

kept = {}
for cap in caps:
    capped = per_photo.assign(points=per_photo["points"].clip(upper=cap))
    kept["cap {}".format(cap)] = capped.groupby(["split", "state_name"])["points"].sum()
kept["no cap"] = per_photo.groupby(["split", "state_name"])["points"].sum()

print("Points kept with a cap per state per photo:")
print(pd.DataFrame(kept).to_string())

### Purity and coverage per crop size

In [ ]:
for size in crop_sizes:
    purity = "purity_{}".format(size)
    coverage = "coverage_{}".format(size)
    summary = points.groupby(["split", "state_name"]).agg(
        purity_p10=(purity, lambda values: values.quantile(0.10)),
        purity_p25=(purity, lambda values: values.quantile(0.25)),
        purity_median=(purity, "median"),
        coverage_median=(coverage, "median"),
    )
    print("Crop size {} px:".format(size))
    print(summary.round(3).to_string())

### What each purity threshold keeps

In [ ]:
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

for size in crop_sizes:
    purity = "purity_{}".format(size)
    kept = {}
    for threshold in thresholds:
        passed = points[points[purity] >= threshold]
        kept["purity >= {}".format(threshold)] = passed.groupby(["split", "state_name"]).size()
    kept["all points"] = points.groupby(["split", "state_name"]).size()
    print("Crop size {} px, points kept:".format(size))
    print(pd.DataFrame(kept).fillna(0).astype(int).to_string())

**Observation**: